In [1]:
SPARK_CONNECT_HOST = "localhost" # if running via an SSH tunnel
#SPARK_CONNECT_HOST = "spark-connector-0-driver-svc" # for running in the Kubernetes cluster

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("test").remote(f"sc://{SPARK_CONNECT_HOST}:15002").getOrCreate()
print("session created")

session created


Take patient data from AWS and bed occupancy data from GCP and join them.

Note that these tables need to be registered first - see instructions in the docs at [README.md](https://github.com/emrys-health/afon-core/blob/main/afon-infra/README.md).

*However*, if you re-create the synthetic data using:
```shell
cd afon-samples/
export MAVEN_OPTS="--add-exports=java.base/sun.nio.ch=ALL-UNNAMED"
mvn -Dexec.mainClass=health.emrys.afon.samples.PatientDataMain exec:java
```

In [ ]:
NAMESPACE = "samples2"
patients = spark.read.table(f"aws.{NAMESPACE}.patients")
occupancy = spark.read.table(f"gcp.{NAMESPACE}.occupancy")
data = patients.join(occupancy, patients.id == occupancy.id)
data.show()

In [14]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

featuresColName = "features"
assembler = (VectorAssembler()
        .setInputCols(["age", "numDays"])
        .setOutputCol(featuresColName))
df = assembler.transform(data)

r1 = Correlation.corr(df, "features").head()
print("Pearson correlation matrix:\n" + str(r1[0]))

Pearson correlation matrix:
DenseMatrix([[1.        , 0.64679249],
             [0.64679249, 1.        ]])
